# Trace Twins — BASELINE notebook

Run top to bottom. Replace the `Submission` below's `score_A`/`score_B` with your real methods**, then build and submit `submission.pkl`.

### 1. Setup (unzip the train data)

In [1]:
# The train data is in this notebook's folder as train_data.zip — unzip it.
!unzip -o train_data\(5\).zip          # -> public_traces.csv

Archive:  train_data(5).zip
  inflating: public_traces.csv       


### 2. Your `Submission` (edit `score_A`/`score_B`)

In [1]:
import io, pickle
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

class Submission:
    def __init__(self):
        # Load or train your models here. The baseline currently needs nothing.
        pass

    # ===== DO NOT EDIT: the cloud calls this to collect your scores =====
    def __call__(self, data: bytes) -> bytes:
        req = pickle.loads(data)
        fn = self.score_A if req["part"] == "A" else self.score_B
        scores = fn(req["windows"], req["pairs"])
        buf = io.BytesIO(); np.save(buf, np.asarray(list(scores), dtype=np.float64))
        return buf.getvalue()
    # ===================================================================    
    def extract_feat(self, corpus):
        w_length = [len(x) for x in corpus]
        vect_alph = []
    
        for w in corpus:
            vf = np.zeros(26)
            for c in w.lower():
                if 'a' <= c <= 'z':
                    vf[ord(c) - ord('a')] += 1
            vect_alph.append(vf)
            
        vect_alph = np.array(vect_alph)
    
        return w_length, vect_alph

    def extract_structural_features(self, windows):
        num_windows = len(windows)
        
        feat_unigrams = np.zeros((num_windows, 100))
        feat_bigrams = np.zeros((num_windows, 150))

        feat_unique = np.zeros((num_windows, 1))
        
        for idx, w in enumerate(windows):

            uni_counts = sorted(Counter(w).values(), reverse=True)
            feat_unigrams[idx, :min(len(uni_counts), 100)] = uni_counts[:100]
            feat_unique[idx, 0] = len(uni_counts)
            
            bigrams = [(w[k], w[k+1]) for k in range(len(w) - 1)]
            bi_counts = sorted(Counter(bigrams).values(), reverse=True)
            feat_bigrams[idx, :min(len(bi_counts), 150)] = bi_counts[:150]
            

        features = np.hstack([feat_unigrams, feat_bigrams, feat_unique])
        norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-9
        return features / norms

    def score_A(self, windows, pairs):
        # Part A (real names): placeholder (AUC 0.5 -> 0 pts). REPLACE with a real method.
        corpus = [" ".join(w) for w in windows]
        vectorizer = TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True)
        X = vectorizer.fit_transform(corpus)

        w_length, alph = self.extract_feat(corpus)
        struct_feats = self.extract_structural_features(windows)
        
        scores = []
        for p in pairs:
            score = X[p[0]].dot(X[p[1]].T)[0, 0]# + 0.2 * float(np.dot(struct_feats[p[0]], struct_feats[p[1]]))   #[0, 0]
            scores.append(score)
                        
        return scores

    def score_B(self, windows, pairs):
        corpus = [" ".join(w) for w in windows]
        vectorizer = TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True)
        X = vectorizer.fit_transform(corpus)
        
        struct_feats = self.extract_structural_features(windows)

        
        scores = []
        for p in pairs:
            score = X[p[0]].dot(X[p[1]].T)[0, 0] + 0.2 * float(np.dot(struct_feats[p[0]], struct_feats[p[1]]))   #[0, 0]
            scores.append(score)
                        
        return scores


### 3. Train your solution (run once)

In [ ]:
sol = Submission()   # loads/trains everything

### 4. (optional) Estimate your score locally

In [2]:
# OPTIONAL local check — self-contained; mirrors the grader (disjoint Part A / Part B
# programs, per-window scramble for B, 50+50 bands). Needs only public_traces.csv.
import csv, random
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

WINDOW = 200; WPP = 8
def _load(p):
    out=[]
    with open(p) as f:
        r=csv.reader(f); next(r)
        for pid,cat,toks in r: out.append({"program_id":int(pid),"category":cat,"tokens":toks.split()})
    return out
def _windows(traces):
    out=[]
    for tr in traces:
        s=tr["tokens"]; n=(len(s)//WINDOW)*WINDOW
        out.extend([{"program_id":tr["program_id"],"category":tr["category"],"wid":j//WINDOW,
                     "tokens":s[j:j+WINDOW]} for j in range(0,n,WINDOW)][:WPP])
    return out
def _pairs(ws,n,seed):
    rng=random.Random(seed); bc=defaultdict(list); bp=defaultdict(list)
    for k,w in enumerate(ws): bc[w["category"]].append(k); bp[w["program_id"]].append(k)
    cats=sorted(bc); multi=[p for p in bp if len(bp[p])>=2]; P=[]; L=[]
    while len(P)<n:
        if rng.random()<0.5:
            p=rng.choice(multi); a,b=rng.sample(bp[p],2); P.append((a,b)); L.append(1)
        else:
            pool=bc[rng.choice(cats)]
            for _ in range(50):
                a,b=rng.sample(pool,2)
                if ws[a]["program_id"]!=ws[b]["program_id"]: P.append((a,b)); L.append(0); break
    return P,L
def _scramble(ws,off):
    vocab=sorted({t for w in ws for t in w["tokens"]}); out=[]
    for w in ws:
        r=random.Random((w["program_id"]*1_000_000+w["wid"])^off); sh=list(vocab); r.shuffle(sh)
        m=dict(zip(vocab,sh)); out.append([m[t] for t in w["tokens"]])
    return out

tr=_load("public_traces.csv")
by_prog={}
for t in tr: by_prog.setdefault(t["program_id"], t)
ids=sorted(by_prog); random.Random(7).shuffle(ids)
val=ids[:int(len(ids)*0.30)]; h=len(val)//2
wA=_windows([by_prog[i] for i in val[:h]]); WA=[w["tokens"] for w in wA]; pA,lA=_pairs(wA,3000,101)
wB=_windows([by_prog[i] for i in val[h:]]); WB=_scramble(wB,303); pB,lB=_pairs(wB,3000,202)
pts=lambda a,b: max(0.0, min(50.0,(a-0.5)/b*50.0))
aucA=roc_auc_score(lA, sol.score_A(WA,pA)); aucB=roc_auc_score(lB, sol.score_B(WB,pB))
print(f"Part A: AUC {aucA:.3f} -> {pts(aucA,0.34):.1f}/50")
print(f"Part B: AUC {aucB:.3f} -> {pts(aucB,0.28):.1f}/50")
print(f"ESTIMATED TOTAL ~ {pts(aucA,0.34)+pts(aucB,0.28):.1f}/100  (secret set differs slightly)")

NameError: name 'sol' is not defined

In [ ]:
['getsystemwindowsdirectoryw', 'loadstringa', 'getsystemwindowsdirectoryw', 'regenumkeyexw', 'ntopenmutant', 'regenumkeyexw', 'ntopenmutant', 'ntopenmutant', 'ntopenmutant', 'ntopenmutant', 'ntsetvaluekey', 'ntsetvaluekey', 'ntsetvaluekey', 'getsystemwindowsdirectoryw', 'regenumkeyexw', 'ntopenmutant', 'getsystemwindowsdirectoryw', 'ntwritevirtualmemory', 'cocreateinstanceex', 'getsystemwindowsdirectoryw', 'getsystemwindowsdirectoryw', 'regsetvalueexa', 'regsetvalueexa', 'cocreateinstanceex', 'createactctxw', 'httpsendrequesta', 'gettemppathw', 'createactctxw', 'httpsendrequesta', 'gettemppathw', 'openscmanagera', 'setendoffile', 'ntmapviewofsection', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'getsystemwindowsdirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'getsystemwindowsdirectoryw', 'getsystemwindowsdirectoryw', 'seterrormode', 'removedirectoryw', 'getsystemwindowsdirectoryw', 'seterrormode', 'removedirectoryw', 'getsystemwindowsdirectoryw', 'seterrormode', 'removedirectoryw', 'getsystemwindowsdirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'seterrormode', 'removedirectoryw', 'seterrormode', 'seterrormode', 'seterrormode', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw', 'removedirectoryw']
['enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', '__exception__', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', 'enumwindows', 'enumwindows', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', '__exception__', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', 'enumwindows', 'enumwindows', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', '__exception__', '__exception__', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', '__exception__', 'enumwindows', '__exception__', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows', 'enumwindows']
['internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer', 'rtldecompressbuffer', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer', 'rtldecompressbuffer', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer', 'rtldecompressbuffer', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer', 'rtldecompressbuffer', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'internetqueryoptiona', 'rtldecompressbuffer', 'internetqueryoptiona', 'rtldecompressbuffer', 'rtldecompressbuffer']


### 5. Build submission.pkl  (run LAST)

In [6]:
WB[9]

['shutdown',
 'getvolumenameforvolumemountpointw',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'shutdown',
 'getvolumenameforvolumemountpointw',
 'shutdown',
 'shutdown',
 'ntcreatemutant',
 'findwindowa',
 'createthread',
 'shellexecuteexw',
 'writeprocessmemory',
 'ntquerysysteminformation',
 'ntcreatefile',
 'coinitializesecurity',
 'ntsetinformationfile',
 'internetconnecta',
 'findwindowa',
 'findwindowa',
 'writeprocessmemory',
 'ntquerysysteminformation',
 'createthread',
 'shellexecuteexw',
 'ntcreatefile',
 'ntsetinformationfile',
 'ntcreatefile',
 'coinitializesecurity',
 'ntsetinformationfile',
 'ntsetinformationfile',
 'createthread',
 'shellexecuteexw',
 'ntcreatefile',
 'ntsetinformationfile',
 'ntcreatefile',
 'coinitializesecurity',
 'ntsetinformationfile',
 'ntsetinformationfile',
 'ntcreatefile',
 'ntcreatefile',
 'ioctlsocket',
 'ioctlsocket',
 'ntcreatefile',
 'coinitializesecurity',
 'ntsetin

In [9]:
# Build submission.pkl  (this is what you upload as the Output)
import cloudpickle, os
# `sol` was trained above. Re-run the train cell first if you restarted the kernel.
with open("submission.pkl", "wb") as f:
    cloudpickle.dump(sol, f)          
mb = os.path.getsize("submission.pkl") / 1e6
print(f"wrote submission.pkl  ({mb:.1f} MB)  -- must be < 50 MB")
assert mb < 50, "too big: cap model size (fewer trees / depth)"

wrote submission.pkl  (0.0 MB)  -- must be < 50 MB


### 6. Submit
Click the **🦆 Submit to Judge** button in the toolbar and choose `submission.pkl` as the Output.